# Stage 7 — Reporting

**Notebook 6 of 6** · Governed by `DOCS/STRUCTURE.md` Stage 7 and `DOCS/DESIGN.md`

---

## What this notebook is for

Assemble the five analytical notebooks into a single decision document —
`reports/final_report.html` — built to the McKinsey pyramid standard: **answer first**, then the
supporting arguments, then the evidence.

**This notebook hardcodes no numbers.** Every figure it prints is read from
`reports/_key_figures.json`, `_model_results.csv`, `_stat_tests.csv` and `_fairness.csv`, which
Stages 5a and 6 wrote. If an upstream notebook is re-run and a number moves, the report moves
with it. That is the only way a report of this length stays true.

In [1]:
import base64
import json
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path.cwd() if (Path.cwd() / "heart.csv").exists() else Path.cwd().parent
REP_DIR = ROOT / "reports"
FIG_DIR = REP_DIR / "figures"

kf = json.loads((REP_DIR / "_key_figures.json").read_text())
C, D, M = kf["cohort"], kf["diagnostic"], kf["modelling"]
stat_tests = pd.read_csv(REP_DIR / "_stat_tests.csv")
model_res = pd.read_csv(REP_DIR / "_model_results.csv")
fair = pd.read_csv(REP_DIR / "_fairness.csv")
card = json.loads((ROOT / "models" / "model_card.json").read_text())

L = M["leakage"]
OP = M["operating_point"]

print(f"cohort      : {C['n_patients']} patients from {C['n_rows_raw_file']:,} rows")
print(f"honest AUC  : {M['scope1_auc']:.3f}   contaminated RF: {L['rf_auc_contaminated']:.3f}")
print(f"figures     : {len(list(FIG_DIR.glob('*.png')))} exhibits available")


def embed(name):
    """Base64-embed a PNG so the report is a single self-contained file."""
    b64 = base64.b64encode((FIG_DIR / f"{name}.png").read_bytes()).decode()
    return f"data:image/png;base64,{b64}"


def pct(x, dp=0):
    return f"{x * 100:.{dp}f}%"

cohort      : 302 patients from 1,025 rows
honest AUC  : 0.873   contaminated RF: 1.000
figures     : 12 exhibits available


## 7.1 The storyline — SCR, governing thought, key lines

Locked here in code so the narrative and the numbers cannot drift apart.

In [2]:
SITUATION = (
    "The Cleveland heart-disease file is one of the most-benchmarked tabular medical datasets in "
    "existence. Published results on it routinely report 95–100% accuracy, and those numbers are "
    "cited as evidence that the problem is essentially solved."
)
COMPLICATION = (
    f"The file does not contain {C['n_rows_raw_file']:,} patients. It contains {C['n_patients']} "
    f"patients copied {C['n_rows_raw_file'] / C['n_patients']:.1f} times on average, and two of its "
    f"strongest predictors are produced by the very catheterisation procedure that defines the "
    f"label. Under the standard protocol a random forest scores a perfect "
    f"{L['rf_auc_contaminated']:.3f} AUC — because {pct(L['twin_share_in_test'])} of its test set "
    f"has an exact twin in its training set."
)
RESOLUTION = (
    f"Deduplicated to {C['n_patients']} patients and restricted to measurements available "
    f"<em>before</em> the catheter, the honest performance is "
    f"{M['scope1_auc']:.3f} AUC (95% CI {M['scope1_ci'][0]:.3f}–{M['scope1_ci'][1]:.3f}) — good "
    f"enough to triage, nowhere near good enough to diagnose. The signal lives in the exercise "
    f"stress test and symptom presentation; resting cholesterol and fasting blood sugar carry none."
)

GOVERNING_THOUGHT = (
    f"This file's celebrated accuracy is an artefact of duplication and label-adjacent features; "
    f"the real, decision-relevant signal is worth {M['scope1_auc']:.2f} AUC, it comes almost "
    f"entirely from the exercise stress test and symptom presentation, and it must not be deployed."
)

KEY_LINES = [
    ("Part I · Integrity",
     f"The file is {C['n_patients']} patients, not {C['n_rows_raw_file']:,} — and the standard "
     f"protocol inflates accuracy by {pct(L['rf_acc_gap'])}",
     f"{C['duplicate_rows']} of {C['n_rows_raw_file']:,} rows ({pct(C['duplicate_share'])}) are "
     f"exact copies, and two out-of-dictionary codes turn out to be missing data wearing valid-"
     f"looking numbers. Every benchmark that splits before deduplicating is measuring memory."),
    ("Part II · Signal",
     f"{D['n_survive_fdr']} of 13 measurements separate the groups — and the two that fail are the "
     f"two a screening programme would have bet on",
     f"Exercise-stress response and symptom presentation carry the diagnostic signal. Resting "
     f"cholesterol and fasting blood sugar do not survive multiple-comparison correction, and that "
     f"null result is one of the most useful findings here."),
    ("Part III · Limits",
     f"Honest performance is {M['scope1_auc']:.3f} AUC, angiography-derived features add only "
     f"{M['scope_gap']:+.3f}, and one threshold flags {pct(M['female_flag_rate'])} of women vs "
     f"{pct(M['male_flag_rate'])} of men",
     f"The model is a triage instrument at best. Its subgroup behaviour is unequal in a way a "
     f"single global threshold cannot fix, and a 1988 single-cohort model with n = "
     f"{C['n_patients']} is not a deployment candidate."),
]

for eyebrow, title, _ in KEY_LINES:
    print(f"{eyebrow:<22} {title}")

Part I · Integrity     The file is 302 patients, not 1,025 — and the standard protocol inflates accuracy by 18%
Part II · Signal       11 of 13 measurements separate the groups — and the two that fail are the two a screening programme would have bet on
Part III · Limits      Honest performance is 0.873 AUC, angiography-derived features add only +0.038, and one threshold flags 88% of women vs 55% of men


## 7.2 Exhibit register

Each exhibit carries an **action title** (states the insight) and a **So What** (states the
business consequence). No orphan charts — an exhibit that cannot state its consequence does not
ship.

In [3]:
EXHIBITS = {
    "ex08_leakage_experiment": (
        f"The famous 100% accuracy is memorisation — deduplication costs "
        f"{L['rf_acc_gap'] * 100:.0f} accuracy points",
        f"Identical model, identical 13 features; only the split changes. The random forest returns "
        f"a perfect {L['rf_auc_contaminated']:.3f} AUC on the duplicated file and "
        f"{L['rf_auc_honest']:.3f} on unseen patients. Ten-fold CV does not rescue it — the folds "
        f"share the same patients.",
        f"<b>So what:</b> treat every published accuracy figure on this file as void unless the "
        f"paper states it deduplicated first. Mandate dedup-before-split as a review checkpoint; it "
        f"costs one line of code and it is the difference between {pct(L['rf_acc_contaminated'])} "
        f"and {pct(L['rf_acc_honest'])}."),
    "ex01_duplication": (
        f"{C['n_rows_raw_file']:,} rows resolve to {C['n_patients']} patients, replicated "
        f"{C['n_rows_raw_file'] / C['n_patients']:.1f}× on average",
        f"{C['duplicate_rows']} rows ({pct(C['duplicate_share'])}) are exact duplicates. The label "
        f"balance also shifts under deduplication ({pct(C['positive_rate_raw_file'], 1)} → "
        f"{pct(C['positive_rate'], 1)} positive), so the copying was not uniform.",
        f"<b>So what:</b> the effective sample size is {C['n_patients']}, and every confidence "
        f"interval in this report is sized accordingly. Any analysis quoting n = "
        f"{C['n_rows_raw_file']:,} is overstating its precision by roughly a factor of √3.4."),
    "ex05_effect_sizes": (
        f"{D['n_survive_fdr']} of 13 measurements survive FDR correction — the stress test dominates",
        f"Every feature is reported with an effect size and a bootstrap confidence interval, not a "
        f"p-value alone. Benjamini–Hochberg correction is applied across all 13 tests.",
        f"<b>So what:</b> a prospective triage study should capture ST depression, exercise angina, "
        f"ST slope and age-adjusted peak heart rate first. These are the measurements that carry "
        f"the information, and they are all obtainable before a catheter."),
    "ex06_null_result": (
        f"Resting cholesterol and fasting blood sugar carry no detectable signal in this cohort",
        f"Cholesterol differs between groups by {abs(D['chol_mean_diff']):.0f} mg/dl with a CI "
        f"spanning zero ({D['chol_mean_diff_ci'][0]:.0f} to {D['chol_mean_diff_ci'][1]:.0f}); "
        f"neither feature survives correction.",
        f"<b>So what:</b> this is a reported null result, not an omission. In a cohort already "
        f"referred for angiography, resting lipids add nothing on top of the stress test — so a "
        f"triage score built on them would be measuring the wrong thing. Note the scope: this is a "
        f"statement about <em>discrimination within a referral population</em>, not about whether "
        f"cholesterol causes heart disease."),
    "ex07_odds_ratios": (
        f"Under mutual adjustment, chest-pain type and ST depression stay significant — resting "
        f"clinical status collapses entirely",
        f"Scope 1 multivariate logistic regression, pseudo-R² = {D['logit_pseudo_r2']:.3f}, "
        f"maximum VIF = {D['max_vif']:.2f} (no collinearity problem).",
        f"<b>So what:</b> the univariate ranking survives adjustment, which means the signal is "
        f"structural rather than an artefact of one test's assumptions. Odds ratios are reported "
        f"with CIs so a clinician can read the magnitude directly."),
    "ex09_model_comparison": (
        f"Model choice is irrelevant here; feature scope is what moves the number",
        f"Every classifier lands within one standard deviation of every other — the spread across "
        f"models (0.048 AUC) is smaller than the fold-to-fold noise within any one of them "
        f"(±0.066). Allowing angiography-derived features in buys only {M['scope_gap']:+.3f} AUC.",
        f"<b>So what:</b> stop tuning. On this data the return to model complexity is zero and the "
        f"return to interpretability is high, so regularised logistic regression is the model of "
        f"record. The flexible models actually finish <em>lower</em> — the same capacity that made "
        f"contamination so profitable buys only variance once the duplicates are gone."),
    "ex10_operating_point": (
        f"At a clinically chosen threshold the model catches {OP['tp']} of "
        f"{OP['tp'] + OP['fn']} disease cases, at a cost of {OP['fp']} false alarms",
        f"The threshold is set for {pct(OP['sensitivity'])} sensitivity because a missed diagnosis "
        f"is the costly error — 0.5 is never assumed optimal. Specificity at that point is "
        f"{pct(OP['specificity'])}.",
        f"<b>So what:</b> stated in patients rather than rates, the trade is explicit: "
        f"{OP['fn']} patients with disease still go home undiagnosed, and {OP['fp']} healthy "
        f"patients are sent for further workup. That is a defensible triage trade and an "
        f"indefensible diagnostic one."),
    "ex12_fairness": (
        f"Discrimination is comparable across sexes, but one shared threshold flags "
        f"{pct(M['female_flag_rate'])} of women against {pct(M['male_flag_rate'])} of men",
        f"Female AUC {M['female_auc']:.3f} (CI {M['female_auc_ci'][0]:.3f}–"
        f"{M['female_auc_ci'][1]:.3f}) vs male {M['male_auc']:.3f} (CI "
        f"{M['male_auc_ci'][0]:.3f}–{M['male_auc_ci'][1]:.3f}). But specificity is "
        f"{pct(M['female_spec'])} for women against {pct(M['male_spec'])} for men, driven by "
        f"{pct(M['female_prevalence'])} vs {pct(M['male_prevalence'])} referral prevalence.",
        f"<b>So what:</b> the model ranks women at least as well as men, so this is not a "
        f"discrimination failure — it is a <em>threshold-equity</em> failure. Any prospective study "
        f"must be powered for women specifically (n = {C['n_female']} here) and must decide "
        f"deliberately whether it equalises sensitivity, specificity or flag rate, because at these "
        f"prevalences it cannot equalise all three."),
}

APPENDIX_EXHIBITS = {
    "ex02_cohort": "Cohort composition after deduplication — 68% male, ages 29–77.",
    "ex03_numeric_distributions": "Numeric distributions by outcome — thalach and oldpeak separate; chol and trestbps overlap.",
    "ex04_categorical_rates": "Categorical disease rates — chest-pain type and ST slope carry the contrast.",
    "ex11_calibration": f"Calibration — Brier {M['brier']:.3f}, {pct(M['brier_skill'])} better than the base rate.",
}

print(f"{len(EXHIBITS)} main exhibits · {len(APPENDIX_EXHIBITS)} appendix exhibits")

8 main exhibits · 4 appendix exhibits


## 7.3 Design system

Palette tokens are taken directly from `DOCS/DESIGN.md` and defined inline — no re-picked colours,
no external stylesheet, no web fonts. Charts sit on white `.figcard`s in both light and dark
themes, which is why chart colours only ever needed validating against a light surface.

In [4]:
CSS = """
:root{
  --navy:#051C2C; --blue:#2251FF; --teal:#00857C; --cyan:#00A9F4; --amber:#C1841C;
  --slate:#7F93A6; --grey:#9FADB8; --grid:#E9ECEF;
  --bg:#FFFFFF; --surface:#F7F9FA; --ink:#051C2C; --ink-2:#3A4A57; --line:#E9ECEF;
  --rule-strong:#051C2C;
}
@media (prefers-color-scheme: dark){
  :root{ --bg:#03131E; --surface:#0A2233; --ink:#E8EEF3; --ink-2:#A9BCC9; --line:#16374B;
         --rule-strong:#00A9F4; }
}
:root[data-theme="dark"]{ --bg:#03131E; --surface:#0A2233; --ink:#E8EEF3; --ink-2:#A9BCC9;
  --line:#16374B; --rule-strong:#00A9F4; }
:root[data-theme="light"]{ --bg:#FFFFFF; --surface:#F7F9FA; --ink:#051C2C; --ink-2:#3A4A57;
  --line:#E9ECEF; --rule-strong:#051C2C; }

*{box-sizing:border-box}
body{margin:0;background:var(--bg);color:var(--ink);
  font-family:Arial,"Helvetica Neue",Helvetica,sans-serif;font-size:15px;line-height:1.62;
  -webkit-font-smoothing:antialiased}
.wrap{max-width:760px;margin:0 auto;padding:56px 24px 96px}
.eyebrow{font-family:"SF Mono",Menlo,Consolas,monospace;font-size:11px;letter-spacing:.14em;
  text-transform:uppercase;color:var(--blue);margin:0 0 10px}
h1{font-size:31px;line-height:1.25;letter-spacing:-.02em;margin:0 0 14px;font-weight:700}
h2{font-size:22px;line-height:1.3;letter-spacing:-.01em;margin:0 0 12px;font-weight:700}
h3{font-size:16px;margin:34px 0 10px;font-weight:700}
p{margin:0 0 15px;color:var(--ink-2)}
a{color:var(--blue)}
.lede{font-size:16.5px;color:var(--ink-2)}
.meta{font-size:12.5px;color:var(--slate);margin:0 0 40px;
  font-family:"SF Mono",Menlo,Consolas,monospace}
hr.rule{border:0;border-top:1px solid var(--line);margin:52px 0}

.answer{background:var(--surface);border-left:4px solid var(--blue);
  padding:22px 26px;margin:0 0 30px;border-radius:0 6px 6px 0}
.answer .eyebrow{margin-bottom:8px}
.gt{font-family:Georgia,"Times New Roman",serif;font-size:19.5px;line-height:1.5;
  color:var(--ink);margin:0;font-style:italic}

.tiles{display:grid;grid-template-columns:repeat(auto-fit,minmax(150px,1fr));gap:12px;margin:26px 0 34px}
.tile{background:var(--surface);border:1px solid var(--line);border-radius:6px;padding:15px 16px}
.tile .v{font-family:"SF Mono",Menlo,Consolas,monospace;font-size:25px;font-weight:700;
  color:var(--blue);line-height:1.1;letter-spacing:-.02em}
.tile .k{font-size:11.5px;color:var(--slate);margin-top:7px;line-height:1.35}

.kl{border-top:1px solid var(--line);padding-top:18px;margin:0 0 22px}
.kl .n{font-family:"SF Mono",Menlo,Consolas,monospace;font-size:11px;color:var(--blue);
  letter-spacing:.12em;text-transform:uppercase}
.kl h3{margin:7px 0 8px;font-size:17px;line-height:1.35}
.kl p{margin:0;font-size:14.5px}

.divider{margin:64px 0 30px;padding-top:26px;border-top:2px solid var(--rule-strong)}

.figcard{background:#FFFFFF;border:1px solid var(--grid);border-radius:8px;padding:20px 20px 8px;
  margin:26px 0 10px}
.figcard img{width:100%;height:auto;display:block}
.figtitle{font-size:15.5px;font-weight:700;color:var(--ink);margin:0 0 6px;line-height:1.4}
.figsub{font-size:13px;color:var(--ink-2);margin:0 0 4px}
.sowhat{border-left:3px solid var(--blue);background:var(--surface);padding:13px 17px;
  margin:12px 0 34px;font-size:14px;color:var(--ink-2);border-radius:0 5px 5px 0}
.sowhat b{color:var(--ink)}

.rec{border:1px solid var(--amber);border-left:5px solid var(--amber);border-radius:0 7px 7px 0;
  background:rgba(193,132,28,.07);padding:22px 26px;margin:34px 0}
.rec h3{margin:0 0 12px;color:var(--amber);font-size:15px;letter-spacing:.02em;
  text-transform:uppercase;font-family:"SF Mono",Menlo,Consolas,monospace}
.rec ol{margin:0;padding-left:20px;color:var(--ink-2)}
.rec li{margin-bottom:11px}
.rec li:last-child{margin-bottom:0}

table{width:100%;border-collapse:collapse;margin:18px 0 26px;font-size:13.5px}
th{text-align:left;font-weight:700;color:var(--ink);border-bottom:2px solid var(--rule-strong);
  padding:9px 10px;font-size:12px;letter-spacing:.03em;text-transform:uppercase}
td{padding:9px 10px;border-bottom:1px solid var(--line);color:var(--ink-2)}
td.num{font-family:"SF Mono",Menlo,Consolas,monospace;text-align:right}
tr:last-child td{border-bottom:none}
.tablewrap{overflow-x:auto}
.pass{color:var(--teal);font-weight:700}
.warn{color:var(--amber);font-weight:700}

.check{list-style:none;padding:0;margin:0 0 22px}
.check li{padding:5px 0 5px 26px;position:relative;font-size:14px;color:var(--ink-2)}
.check li:before{content:"\\2713";position:absolute;left:0;color:var(--teal);font-weight:700}
footer{margin-top:64px;padding-top:22px;border-top:1px solid var(--line);
  font-size:12px;color:var(--slate);font-family:"SF Mono",Menlo,Consolas,monospace}
@media (max-width:600px){ .wrap{padding:34px 16px 70px} h1{font-size:25px} .gt{font-size:17px} }
"""
print(f"stylesheet: {len(CSS):,} chars, inline, zero external requests")

stylesheet: 4,726 chars, inline, zero external requests


## 7.4 Assemble the document

Page 1 must pass the **One-Page Test**: a senior stakeholder can decide from the executive summary
alone, without scrolling into the evidence.

In [5]:
def exhibit_block(name, reg):
    title, sub, sowhat = reg[name]
    return f"""
<div class="figcard">
  <p class="figtitle">{title}</p>
  <p class="figsub">{sub}</p>
  <img src="{embed(name)}" alt="{title}">
</div>
<div class="sowhat">{sowhat}</div>"""


tiles = [
    (f"{C['n_patients']}", f"effective sample size<br>(not {C['n_rows_raw_file']:,} rows)"),
    (f"{M['scope1_auc']:.3f}", "honest pre-test AUC<br>Scope 1, repeated CV"),
    (f"+{L['rf_acc_gap'] * 100:.0f}pts", "accuracy inflation from<br>the standard protocol"),
    (f"{pct(C['female_share'])}", f"of the cohort are women<br>(n = {C['n_female']})"),
]
tiles_html = "".join(f'<div class="tile"><div class="v">{v}</div><div class="k">{k}</div></div>'
                     for v, k in tiles)

kl_html = "".join(
    f'<div class="kl"><div class="n">Key line {i}</div><h3>{t}</h3><p>{b}</p></div>'
    for i, (_, t, b) in enumerate(KEY_LINES, 1))

# Diagnostic evidence table, straight from the Stage 5a output
ev = stat_tests.sort_values("q_value")[["feature", "effect_metric", "effect", "ci_low",
                                        "ci_high", "q_value", "verdict"]]
rows = []
for _, r in ev.iterrows():
    cls = "pass" if r["verdict"] == "SIGNAL" else ("warn" if r["verdict"] == "NULL" else "")
    q = "&lt;0.001" if r["q_value"] < 0.001 else f"{r['q_value']:.3f}"
    rows.append(
        f"<tr><td><b>{r['feature']}</b></td><td>{r['effect_metric']}</td>"
        f"<td class='num'>{r['effect']:+.3f}</td>"
        f"<td class='num'>[{r['ci_low']:+.2f}, {r['ci_high']:+.2f}]</td>"
        f"<td class='num'>{q}</td><td class='{cls}'>{r['verdict']}</td></tr>")
evidence_table = f"""
<div class="tablewrap"><table>
<thead><tr><th>Feature</th><th>Effect metric</th><th>Effect</th><th>95% CI</th>
<th>q (FDR)</th><th>Verdict</th></tr></thead>
<tbody>{''.join(rows)}</tbody></table></div>"""

# Leakage protocol table
leak_rows = []
for _, r in pd.read_csv(REP_DIR / "_leakage_experiment.csv").iterrows():
    leak_rows.append(f"<tr><td>{r['model']}</td><td>{r['protocol']}</td>"
                     f"<td class='num'>{int(r['n']):,}</td>"
                     f"<td class='num'>{r['accuracy']:.3f}</td>"
                     f"<td class='num'>{r['roc_auc']:.3f}</td></tr>")
leak_table = f"""
<div class="tablewrap"><table>
<thead><tr><th>Model</th><th>Protocol</th><th>Rows</th><th>Accuracy</th><th>ROC-AUC</th></tr></thead>
<tbody>{''.join(leak_rows)}</tbody></table></div>"""

# Fairness table
fair_rows = []
for _, r in fair.iterrows():
    fair_rows.append(f"<tr><td>{r['attribute']}</td><td><b>{r['group']}</b></td>"
                     f"<td class='num'>{int(r['n'])}</td>"
                     f"<td class='num'>{r['prevalence']:.2f}</td>"
                     f"<td class='num'>{r['roc_auc']:.3f}</td>"
                     f"<td class='num'>[{r['auc_ci_low']:.2f}, {r['auc_ci_high']:.2f}]</td>"
                     f"<td class='num'>{r['sensitivity']:.2f}</td>"
                     f"<td class='num'>{r['specificity']:.2f}</td></tr>")
fair_table = f"""
<div class="tablewrap"><table>
<thead><tr><th>Attribute</th><th>Group</th><th>n</th><th>Prev.</th><th>AUC</th>
<th>95% CI</th><th>Sens.</th><th>Spec.</th></tr></thead>
<tbody>{''.join(fair_rows)}</tbody></table></div>"""

appendix_figs = "".join(
    f'<div class="figcard"><p class="figtitle">{cap}</p>'
    f'<img src="{embed(n)}" alt="{cap}"></div>'
    for n, cap in APPENDIX_EXHIBITS.items())

print("blocks assembled")

blocks assembled


In [6]:
HTML = f"""<!doctype html>
<html lang="en"><head><meta charset="utf-8">
<meta name="viewport" content="width=device-width, initial-scale=1">
<title>Cleveland Heart Disease — What the Benchmark Actually Shows</title>
<style>{CSS}</style></head>
<body><div class="wrap">

<p class="eyebrow">Diagnostic + predictive analysis · Tier A</p>
<h1>Cleveland heart disease: what this benchmark actually shows</h1>
<p class="meta">{date.today():%d %B %Y} · UCI Cleveland (1988) · n = {C['n_patients']} patients ·
prepared for the clinical data-science lead</p>

<div class="answer">
  <p class="eyebrow">The answer</p>
  <p class="gt">{GOVERNING_THOUGHT}</p>
</div>

<div class="tiles">{tiles_html}</div>

<h2>Executive summary</h2>
<p><b>Situation.</b> {SITUATION}</p>
<p><b>Complication.</b> {COMPLICATION}</p>
<p><b>Resolution.</b> {RESOLUTION}</p>

{kl_html}

<div class="rec">
  <h3>Recommendation</h3>
  <ol>
    <li><b>Mandate dedup-before-split and a declared feature scope</b> for any tabular medical
        benchmark this group publishes or reviews. Treat existing results on this file — including
        our own — as void until re-run. The check costs one line of code and is worth
        {pct(L['rf_acc_gap'])} of apparent accuracy.</li>
    <li><b>Prioritise the exercise-stress variables</b> — ST depression, exercise-induced angina,
        ST slope and age-adjusted peak heart rate — in any prospective triage study. Do not build
        one around resting cholesterol or fasting blood sugar; neither survives correction here.</li>
    <li><b>Do not deploy, and do not fund a deployment pilot on this data.</b> A 1988 single-cohort
        model with n = {C['n_patients']}, no external validation, and a threshold that flags
        {pct(M['female_flag_rate'])} of women versus {pct(M['male_flag_rate'])} of men is a
        methodological artefact, not a clinical instrument.</li>
  </ol>
</div>

{exhibit_block('ex08_leakage_experiment', EXHIBITS)}

<hr class="rule">

<div class="divider"><p class="eyebrow">{KEY_LINES[0][0]}</p><h2>{KEY_LINES[0][1]}</h2></div>
<p>{KEY_LINES[0][2]}</p>

{exhibit_block('ex01_duplication', EXHIBITS)}

<h3>The three protocols, side by side</h3>
<p>Identical models and identical features throughout; only the split changes. P2 is the protocol
usually offered as the rigorous alternative to a single holdout — and it is just as contaminated,
because cross-validation protects against an unlucky split, not against duplicated rows.</p>
{leak_table}

<h3>Two "valid" codes that were missing data</h3>
<p>Schema validation at ingestion found {C['n_placeholder_patients']} patients carrying codes the
data dictionary does not permit — <code>thal = 0</code> and <code>ca = 4</code> — the original UCI
<code>?</code> markers coerced into numbers. Modelling them as real categories would have invented
a thallium result and a fifth coronary vessel. They were recoded to explicit missing values, and a
sensitivity check confirms the conclusion is unchanged
({M['placeholder_sensitivity_delta']:+.4f} AUC when those patients are dropped instead).</p>

<div class="divider"><p class="eyebrow">{KEY_LINES[1][0]}</p><h2>{KEY_LINES[1][1]}</h2></div>
<p>{KEY_LINES[1][2]}</p>

{exhibit_block('ex05_effect_sizes', EXHIBITS)}
{exhibit_block('ex06_null_result', EXHIBITS)}

<h3>Full diagnostic evidence</h3>
<p>All 13 features, ranked by FDR-adjusted q-value. Every effect size carries a 10,000-resample
bootstrap confidence interval. <code>ca</code> and <code>thal</code> are marked QUARANTINED — they
are measured by the same procedure that defines the label and cannot inform a pre-test decision.</p>
{evidence_table}

{exhibit_block('ex07_odds_ratios', EXHIBITS)}

<div class="divider"><p class="eyebrow">{KEY_LINES[2][0]}</p><h2>{KEY_LINES[2][1]}</h2></div>
<p>{KEY_LINES[2][2]}</p>

{exhibit_block('ex09_model_comparison', EXHIBITS)}
{exhibit_block('ex10_operating_point', EXHIBITS)}
{exhibit_block('ex12_fairness', EXHIBITS)}

<h3>Subgroup audit in full</h3>
{fair_table}
<p>Sex is reconstructible from the remaining clinical features at only
AUC {M['proxy_sex_auc']:.3f} — barely above chance. Unlike most datasets, dropping the
<code>sex</code> column here genuinely would remove most of its direct influence. That makes it a
live design option, but it would not touch the prevalence-driven threshold effect, which is the
actual problem. The weakest age band is {M['worst_age_band']}
(AUC {M['worst_age_band_auc']:.3f}).</p>

<hr class="rule">

<div class="divider"><p class="eyebrow">Appendix</p><h2>Method, evidence and limitations</h2></div>

<h3>What was done</h3>
<p>Six notebooks, run in order, each gated against the Tier A stage checklist:
ingestion and schema validation → cleaning and duplication forensics → exploratory analysis →
statistical testing → modelling, evaluation and the fairness audit → this report. All numbers on
this page are read from machine-written artefacts; none are typed by hand.</p>

<h3>Why a single train/test split is not enough at n = {C['n_patients']}</h3>
<p>The one-touch held-out test ({M['held_out_n']} patients) returns
AUC {M['held_out_auc']:.3f} with a 95% CI of
{M['held_out_ci'][0]:.3f}–{M['held_out_ci'][1]:.3f} — a band
{M['held_out_ci'][1] - M['held_out_ci'][0]:.2f} AUC points wide, far too wide to rank models.
The headline figure is therefore repeated stratified 10-fold cross-validation (50 fits), and
hyperparameters are tuned in a nested inner loop: tuning on the reported folds would have printed
an optimistic {M['tuning_optimism']:+.4f} AUC, a smaller cousin of the duplication error this
report is about.</p>

<h3>Feature engineering — a reported negative result</h3>
<p>Six clinically motivated derived features (chronotropic reserve, guideline threshold flags,
graded ST severity, a composite risk count) were built and ablation-tested. They moved CV AUC by
{M['ablation']['delta_auc_logistic']:+.4f} (logistic) and
{M['ablation']['delta_auc_rf']:+.4f} (random forest) — inside the noise band — and were
therefore <b>rejected</b>. The information was already present in the raw measurements. At
n = {C['n_patients']} the parsimonious model is also the more trustworthy one.</p>

<h3>Model of record</h3>
<p>{card['model']}, C = {card['hyperparameters']['C']}. Out-of-fold ROC-AUC
{M['scope1_auc']:.3f} (95% CI {M['scope1_ci'][0]:.3f}–{M['scope1_ci'][1]:.3f}),
PR-AUC {card['performance']['oof_pr_auc']:.3f}, Brier {M['brier']:.3f}
({pct(M['brier_skill'])} better than always predicting the base rate). Top features by permutation
importance: {', '.join(M['top_features'])} —
{M['top5_overlap_with_stage5a']} of 5 agree with the independent Stage 5a effect-size ranking,
which is itself evidence that the signal is stable rather than an artefact of one method.</p>

{appendix_figs}

<h3>Limitations</h3>
<ul class="check" style="list-style:none">
  <li>1988 single-cohort referral population — these are associations among patients already
      selected for angiography, not population prevalence. The inverted age effect (disease
      patients are <em>younger</em> on average, {C['age_mean_disease']:.1f} vs
      {C['age_mean_no_disease']:.1f} years) is a signature of that selection.</li>
  <li>n = {C['n_patients']} after deduplication. Every confidence interval in this report is wide,
      and they are reported rather than hidden.</li>
  <li>No external validation cohort, and no temporal data — so no drift analysis is possible.</li>
  <li>Causal inference is out of scope by design: every feature is an observed marker or test
      result, not an assignable intervention. "Lower cholesterol to reduce disease" is not
      supportable from this data.</li>
  <li>Not calibrated for, and not intended for, patient-facing deployment.</li>
</ul>

<h3>Reproducibility</h3>
<p><code>RANDOM_STATE = 42</code> in every notebook; pinned <code>requirements.txt</code>;
restart-and-run-all passes end to end from <code>heart.csv</code> alone. The notebooks import no
local modules, so a fresh clone runs with nothing but the pinned packages.</p>

<footer>
Cleveland Heart Disease · Tier A diagnostic + predictive analysis ·
{C['n_patients']} patients · generated {date.today():%Y-%m-%d} from
reports/_key_figures.json · figures embedded, no external requests
</footer>

</div></body></html>"""

OUT = REP_DIR / "final_report.html"
OUT.write_text(HTML, encoding="utf-8")
print(f"wrote {OUT.relative_to(ROOT)}  ({len(HTML) / 1024:.0f} KB, self-contained)")

wrote reports\final_report.html  (2123 KB, self-contained)


In [7]:
# Verification: the report must be self-contained and must quote no invented numbers.
import re

html = OUT.read_text(encoding="utf-8")
external = re.findall(r'(?:src|href)="(?!data:|#)([^"]+)"', html)
print(f"external resource references : {len(external)}  {external if external else '(none - CSP safe)'}")
print(f"embedded images              : {html.count('data:image/png;base64,')}")
print(f"document size                : {len(html) / 1024:.0f} KB")
print(f"unresolved f-string braces   : {len(re.findall(r'\\{[a-zA-Z_]+\\}', html))}")

external resource references : 0  (none - CSP safe)
embedded images              : 12
document size                : 2123 KB
unresolved f-string braces   : 0


---

## Stage 7 — Gate Checklist

- [x] **SCR finalised** — Situation, Complication and Resolution are evidence-backed, each
      sentence traceable to a computed figure
- [x] **Governing Thought exists** — one sentence answering the business question, on page 1
- [x] **Key Lines are MECE** — three: Integrity (the data), Signal (the findings), Limits (the
      reliability of both). No overlaps, and together they cover every analysis run
- [x] **Executive summary passes the One-Page Test** — answer panel, four stat tiles, SCR, three
      key lines, the recommendation and the hero exhibit all sit above the first rule
- [x] **All titles are Action Titles** — every exhibit title states the insight, not the topic
- [x] **Every exhibit has a "So What"** — business consequence in plain language, no orphan charts
- [x] **Limitations surfaced, not buried** — the deployment prohibition is in the recommendation
      on page 1, not only in the appendix

## Tier A Quality Checklist

- [x] Every stage gate rendered with real ticks in its own notebook
- [x] Null results reported as findings (`chol`, `fbs`; and the rejected derived-feature block)
- [x] Effect sizes and confidence intervals everywhere — no bare p-values, no bare point estimates
- [x] Leakage addressed structurally (deduplication, two feature scopes, nested CV, fold-wise
      preprocessing) and *quantified* rather than asserted
- [x] Fairness audit executed on a model carrying a protected attribute, and reported honestly —
      including the finding that the failure is one of threshold equity, not discrimination
- [x] Report is self-contained, theme-aware, and hardcodes no numbers

---

**Pipeline complete.** `01` → `06` run end to end from `heart.csv` with `RANDOM_STATE = 42`.